# 第13回 演習：クラスタリング

## 前回からの接続

第12回では、6つの変数を2本の軸に束ね、バイプロット（biplot）で「列（変数）の方向の要約」を読みました。今日は要約する向きが変わります。列ではなく**行**——142か国という一つひとつの個体を、似た者どうしの群れに分けます。

前回の最後に置いた二つの問いが、そのまま今日の宿題です。当てるべき正解ラベルがないのに「うまく分かれた」とどうやって判定するのか。そして、群れはいくつに分ければよいのか。前者には慣性とシルエット（silhouette）係数、後者にはエルボー法（elbow method）という物差しで答えを出していきます。

## 今日の分析目標

**国々を、根拠あるグループ数で、意味あるグループに分けたい。**

この演習では、世界の国々を K-means で分け、エルボー法とシルエット係数でグループ数（K）の根拠を測る流れを、自分の手で動かします。K-means が何を最小化しているのか、シルエット係数がどう「凝集」と「分離」を一つの数にまとめるのかまで踏み込みます。TODOに取り組みながら、最後の「目標に答えられたか」で振り返りましょう。

## 学習ゴール

この回を終えると、次のことができるようになります。

- K-means の4ステップが、慣性という一つの量を下げ続ける手続きであることを、図と数値で説明できる
- 距離で似ているかを決める手法だから標準化（standardization）と対数化を先に済ませる、というその順序の理由を言える
- シルエット係数を $a_i$（凝集）と $b_i$（分離）に分けて読み、ある国の値が高い・負になる意味を説明できる
- エルボーとシルエットが違う K を指したとき、目的を根拠に K を選び、その理由を言葉にできる
- ARI が測れるものと測れないものを区別し、内部指標との使い分けを説明できる


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
try:
    import japanize_matplotlib
except Exception:   # japanize が動かなくなったときの保険：同梱フォントを直接登録する
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None) if spec else None
    if ttf:
        fm.fontManager.addfont(str(ttf))
        plt.rcParams['font.family'] = fm.FontProperties(fname=str(ttf)).get_name()

plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
DATA_DIR = 'https://raw.githubusercontent.com/k0heiun0/applied_exercise/main/shared/data'   # データはこのリポジトリから読み込む
df = pd.read_csv(f'{DATA_DIR}/gapminder.csv')
d = df[df.year == 2007].copy()
d['loggdp'] = np.log10(d['gdpPercap'])
X = StandardScaler().fit_transform(d[['lifeExp', 'loggdp']])
print(f'{len(d)}か国。平均寿命とGDP（対数）で分けます')

### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は色の濃いところを扱います。


In [ ]:
# 図：分析の地図（全14回のどこにいるか）
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
here = {2, 3}
for i, (p, c, x) in enumerate(zip(phases, colors, [0.17, 0.5, 0.83]), start=1):
    on = i in here
    ax.text(x, 0.62, p, ha='center', va='center', fontsize=13 if on else 11,
            color='white', bbox=dict(boxstyle='round,pad=0.6', facecolor=c,
                                     alpha=0.95 if on else 0.25))
for x0, x1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(x1, 0.62), xytext=(x0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


## 1. K-means で3グループに分けて図示

まず K=3 で分け、GDP（対数）と平均寿命の散布図で見ます。

### 深掘り：K-means は何を最小化しているのか——慣性という目的関数と「球状クラスタ」の仮定

**4ステップの正体は「慣性を下げる」繰り返し**　K-means の手順は「①中心をランダムに置く ②各点を最寄りの中心へ割り当てる ③中心をメンバーの重心（centroid）へ動かす ④動かなくなるまで②③を繰り返す」でした。この一見素朴な繰り返しは、じつは一つの量をひたすら小さくしています。それが**慣性（イナーシャ）**——各点から自分のクラスタ中心までの距離の二乗を、全点で足し上げたものです。

$$
\text{慣性} \;=\; \sum_{c=1}^{K}\ \sum_{i \in C_c} \lVert x_i - \mu_c \rVert^2
$$

記号をほどきます。$C_c$ は $c$ 番目のクラスタ（に属する国の集合）、$\mu_c$ はその中心（セントロイド＝メンバーの平均の位置）、$\lVert x_i - \mu_c \rVert^2$ は国 $x_i$ と中心の距離の二乗です。**同じクラスタの中がぎゅっとまとまるほど、この値は小さく**なります。つまり慣性は、良いクラスタリング（clustering）の条件のうち「凝集度（中はぎゅっと）」を数値にしたものです。

**なぜ収束（convergence）するのか**　ステップ②（割り当て）は「中心を固定して、各点を最寄りへ付け替える」——これは慣性を下げるか、少なくとも増やしません。ステップ③（中心更新）は「割り当てを固定して、中心を重心へ動かす」——重心はメンバーまでの距離二乗和を最小にする点なので、これも慣性を下げます。**二つの操作がどちらも慣性を減らすだけ**なので、値は下がり続けてやがて底を打ち、誰も移らなくなる。これが収束の中身で、ふつう数回〜十数回で落ち着くのが速さの理由です。ただし下がり着く先は「その初期値から届く谷底」であって、**全体の最小とは限りません**。初期値が悪ければ浅い谷で止まる——だからコードの `n_init=10` が初期値を10通り試し、いちばん慣性の低い結果を採る保険になっています。どの初期値がどの谷へ落ちるのか、初期値の置き方しだいで最終的な慣性がどこまで悪くなりうるのかを見積もった議論は、初期値の選び方そのものを主題にした k-means++ の原論文（Arthur・Vassilvitskii, 2007）に譲ります。

**なぜ「球状（丸いかたまり）」を仮定することになるのか**　鍵は、ステップ②が**中心までのユークリッド距離（Euclidean distance）**だけで割り当てることです。「中心から等距離の点」を結ぶと**円（高次元では球）**になり、K-means は暗黙に「各クラスタは中心のまわりに丸く、等方的に散らばっている」と決めつけています。だから細長い帯や三日月のように**曲がった／偏った形は苦手**。中心から測るぶん、まっすぐで丸いかたまりしか正しく囲えないのです。このデータで K=3 がうまくいくのは、平均寿命と対数 GDP の平面で、最貧・新興・先進の三つがたまたま**おおむね丸い塊**に分かれているからで、それが成り立つ前提つきの成功だと心得ておきます。

**この節の実データ**　K=3 の三グループは、下の TODO① で確かめるとおり、次のように整理されます（TODO①を正しく解いたときの実行値）。

| グループ | 国数 | 平均寿命 | 平均GDP | 正体 |
|---:|---:|---:|---:|---|
| 0 | 52 | 53.0歳 | 約2,079ドル | 最貧（アフガニスタン等） |
| 1 | 51 | 72.2歳 | 約7,307ドル | 新興・途上（アルバニア等） |
| 2 | 39 | 78.8歳 | 約30,201ドル | 先進（オーストラリア等） |

ラベルを一切与えていないのに、寿命と GDP の距離だけから**発展の三段階**が浮かびました。中心へ近い順にまとめる、というそれだけの仕組みが、社会経済の実感に沿うグループを描き出しています。

**つまずきどころ：距離で分けるから、必ず先に標準化する**　K-means は距離で「似ているか」を判断します。ところが平均寿命は数十のオーダー、一人あたり GDP は数千〜数万のオーダー。素のまま距離を測れば、**桁の大きい GDP だけで距離が決まり**、寿命の違いはほぼ無視されてしまいます。だから第4回の標準化で両者を無単位・分散1にそろえてから分けるのが必須です（コード冒頭の `StandardScaler`）。さらに GDP は先に**対数**をとっています。GDP は数百ドルから数万ドルまで桁で大きく広がるため、生の値では豊かな国どうしの差ばかりが効いてしまう。対数にすると「何倍か」で効くようになり、貧しい国どうしの小さな差も対等に扱えます。標準化と対数化——この二つの下ごしらえが、距離を歪ませないための土台です。


**別の見方：慣性は「クラスタ内の分散」そのもの**　慣性 $\sum_c \sum_{i\in C_c}\lVert x_i-\mu_c\rVert^2$ は、各クラスタの中心まわりのばらつきを、クラスタ横断で足し上げたものです。第11・12回の PCA が「分散を最大に残す軸」を探したのと裏表で、K-means は「クラスタ内の分散を最小にする分け方」を探しています。同じ“散らばり”という物差しが、片や軸を選び、片やグループを分ける——教師なし学習が一貫して分散を手がかりにしていることが見て取れます。さらに距離二乗和を最小化する性質上、K-means は**同じくらいの大きさ・同じくらいの広がりの塊**を好みます。極端に大きい塊と小さい塊が混じると、大きい塊を割って小さい塊を呑み込みがち——「球状」だけでなく「同じくらいの大きさ」も暗黙の前提だと覚えておくと、結果が直感とずれたときに原因の見当がつきます。三日月型のデータを K-means が左右にぶった切ってしまうのも、上下の三日月が“中心から丸く等距離”という前提に反するからで、慣性やシルエットも丸い前提のうえで測るぶん、この手の形では当てになりません。

In [ ]:
# 図：K-means の4つのステップ（実データで、初期値から収束まで追う）
_colors = ['#e63946', '#e9c46a', '#2a9d8f']
_rng = np.random.default_rng(334)
_C0 = X[_rng.choice(len(X), 3, replace=False)]                 # ① 中心を3つ、ランダムに置く
_assign = lambda C: ((X[:, None, :] - C[None, :, :]) ** 2).sum(-1).argmin(1)
_inertia = lambda lab, C: ((X - C[lab]) ** 2).sum()
_lab1 = _assign(_C0)                                           # ② 各点を最寄りの中心へ
_C1 = np.array([X[_lab1 == k].mean(0) for k in range(3)])      # ③ 中心をメンバーの重心へ
_C, _lab, _n = _C1, _lab1, 1                                   # ④ 動かなくなるまで②③を繰り返す
while True:
    _lab = _assign(_C)
    _new = np.array([X[_lab == k].mean(0) for k in range(3)])
    _n += 1
    if np.allclose(_new, _C):
        break
    _C = _new

_fig, _axes = plt.subplots(2, 2, figsize=(10, 7.2), sharex=True, sharey=True)
_panels = [('① 中心を3つ、ランダムに置く', None, _C0, None),
           (f'② 各点を最寄りの中心へ　慣性 {_inertia(_lab1, _C0):.1f}', _lab1, _C0, None),
           (f'③ 中心をメンバーの重心へ　慣性 {_inertia(_lab1, _C1):.1f}', _lab1, _C1, _C0),
           (f'④ ②③を{_n}回で収束　慣性 {_inertia(_lab, _C):.1f}', _lab, _C, None)]
for _ax, (_t, _l, _cen, _prev) in zip(_axes.ravel(), _panels):
    _cs = ['#c9c9c9'] * len(X) if _l is None else [_colors[k] for k in _l]
    _ax.scatter(X[:, 1], X[:, 0], c=_cs, s=18, alpha=0.85, linewidths=0)
    if _prev is not None:                      # ③ だけ、動く前の位置（白抜き）と矢印を重ねる
        _ax.scatter(_prev[:, 1], _prev[:, 0], marker='X', s=110, facecolors='none',
                    edgecolors='#333', linewidths=1.4)
        for _a, _b in zip(_prev, _cen):
            _ax.annotate('', xy=(_b[1], _b[0]), xytext=(_a[1], _a[0]),
                         arrowprops=dict(arrowstyle='->', color='#333', lw=1.8))
    _ax.scatter(_cen[:, 1], _cen[:, 0], marker='X', s=170, c='#1a1a2e',
                edgecolors='white', linewidths=1.2, zorder=5)
    _ax.set_title(_t, fontsize=12)
for _ax in _axes[1]:
    _ax.set_xlabel('一人あたりGDP（対数）の標準化値')
for _ax in _axes[:, 0]:
    _ax.set_ylabel('平均寿命の標準化値')
_fig.suptitle('×＝クラスタ中心。点の色は、そのときの割り当て（灰色＝まだ未割り当て）', fontsize=12, color='#555')
plt.tight_layout(); plt.show()


**この図の読み方**　4枚は左上から右下へ、この節のはじめの深掘りが並べた①〜④の順です。横軸は一人あたり GDP（対数）、縦軸は平均寿命で、どちらも**標準化した値**——K-means が実際に距離を測っている座標です。黒い × が中心（セントロイド）、点の色はそのときの割り当て、灰色は「まだどこにも属していない」を表します。①は142か国から3か国をランダムに選んで中心に据えた状態で、たまたま2つが右上の豊かな側に固まりました。②はその中心へ各点を最寄り割り当てしたところで、右上で緑と黄が入り混じり、左と下の広い範囲がまるごと赤に呑まれています。③では白抜きの ×（動く前の位置）から矢印の先へ、3つの中心がメンバーの重心へ移ります。動いた先で最寄りが変わる点が出るので、また②に戻る——この往復が④です。

**慣性が下がるところを数字で追う**　各パネルの見出しに慣性を出してあります。偏った初期値のままの②は 152.6。中心を重心へ動かしただけの③で 96.3 まで落ち、②③を繰り返して7回目に誰も移らなくなり、**53.8** で止まります。この 53.8 は、2節のエルボー法で K=3 のときに現れる値と同じ数字です。そして④の分け方は、この節の `KMeans(n_clusters=3, random_state=42, n_init=10)` が出す結果と1か国の違いもなく一致します（52・51・39か国、色も対応しています）。ここでは偏った初期値から始めても同じ谷に届きましたが、いつもそうなるとは限らない——だから `n_init=10` で初期値を振り直す保険が要るのです。


In [ ]:
km = KMeans(n_clusters=3, random_state=42, n_init=10)
d['cluster'] = km.fit_predict(X)
colors = ['#e63946', '#e9c46a', '#2a9d8f']
for cl in range(3):
    m = d['cluster'] == cl
    plt.scatter(d.loc[m, 'gdpPercap'], d.loc[m, 'lifeExp'], c=colors[cl], s=30, alpha=0.8, label=f'グループ{cl}')
plt.xscale('log'); plt.xlabel('一人あたりGDP（対数）'); plt.ylabel('平均寿命'); plt.legend()
plt.tight_layout(); plt.show()

### TODO①：各グループの特徴を要約する

各グループの「国数・平均寿命の平均・GDPの平均」を計算して表示し、どのグループが「最貧・新興・先進」かを読み取ってください。

### 深掘り：三つのグループに“名前”をつける——記述統計から意味を読む、その手前のつまずき

**数字から正体を読む**　TODO① で `groupby('cluster')` を使うと、各グループの国数・平均寿命・平均 GDP が並びます（実行値は次のとおり）。グループ0は平均寿命53歳・GDP約2千ドルで最も貧しく短命、グループ2は79歳・約3万ドルで豊かで長寿、グループ1はその中間です。ここに代表国（グループ0はアフガニスタンやアンゴラ、グループ2はオーストラリアやオーストリア）を重ねると、「最貧・新興・先進」という正体がくっきり浮かびます。**クラスタリングが出すのは番号だけ**で、その番号に“最貧”“先進”という意味を与えるのは、記述統計と現実の知識を突き合わせる**人間の仕事**です。分けて終わりにせず、必ず各グループの中身を要約して名前をつける——この一手が、報告で使える分析と使えない分析を分けます。

**つまずきどころ：クラスタ番号そのものに意味はない**　`cluster` の 0・1・2 という番号は、アルゴリズムが内部で振った**ただの通し番号**で、大小や順序に意味はありません。乱数（random number）シード（`random_state`）を変えれば、同じ三グループでも「最貧が0」だったのが「最貧が2」に入れ替わることがあります（ラベル・スイッチング）。だから「グループ0＝最貧」と決め打ちでコードに書くのは危険で、**必ず平均寿命や GDP の中身を見てから**意味づけするのが安全です。同じ理由で、ARI のような一致度の指標は「番号がどう振られたか」に左右されない作りになっています（番号を付け替えただけでスコアが動いては困るからです）。番号は仮の名札、中身こそが本体、と心に留めておきましょう。

In [ ]:
# TODO: d を cluster でグループ化し、国数・平均寿命・GDPの平均を表示してください
# ヒント: d.groupby('cluster').agg(...) や、.mean() が使えます
...

## 2. エルボー法とシルエットで、Kの根拠を探す

### TODO②：エルボー（慣性）とシルエットを計算する

K を 2〜8 まで変えて、①慣性（`.inertia_`）と ②シルエットスコアを計算し、それぞれ表示（またはグラフ化）してください。エルボーの肘と、シルエットのピークは、同じKでしたか？

### 深掘り：シルエット係数の骨子——「自分のグループにしっくり収まっているか」を数式にする

**主役の式**　シルエット係数は、各点（各国）$i$ ごとに「いまのクラスタに気持ちよく収まっているか」を一つの数にします。

$$
s_i \;=\; \frac{b_i - a_i}{\max(a_i,\ b_i)}, \qquad -1 \le s_i \le 1
$$

- $a_i$：**同じクラスタ内**の他の国たちまでの平均距離。小さいほど「仲間の近くにいる」＝**凝集**。
- $b_i$：**最も近い“別”のクラスタ**の国たちまでの平均距離。大きいほど「隣から遠い」＝**分離**。ここで「最も近い別クラスタ」だけを相手に選ぶのがポイントで、いちばん引っ越したくなる隣と比べます。

分子 $b_i - a_i$ は「隣は遠く（$b_i$ 大）、仲間は近い（$a_i$ 小）」ほど大きくなる、収まりの良さそのもの。分母 $\max(a_i,b_i)$ は、それを $-1$〜$1$ の範囲に収めるための割り算です。読み方はシンプルで、**$s_i \approx 1$：くっきり収まっている／$s_i \approx 0$：境界線上でどっちつかず／$s_i < 0$：隣のクラスタのほうが近く、割り当てを間違えている疑い**。凝集（$a_i$）と分離（$b_i$）を一度に見てくれるのがシルエットの強みで、凝集しか測れない慣性を補います。クラスタに1点しかない場合の扱いまで詰めた厳密な定義と、これから触れる他の内部指標との関係は、クラスタ妥当性（validity）指標を横断的に比べたサーベイ論文に譲ります。

**実データで一つ計算してみる（日本）**　K=3 のとき、日本は先進グループ（グループ2）に入ります。日本から先進グループの他国までの平均距離は $a_i \approx 0.393$、いちばん近い別グループ（新興グループ）までの平均距離は $b_i \approx 1.493$。式に入れると

$$
s_{\text{日本}} \;=\; \frac{1.493 - 0.393}{\max(0.393,\ 1.493)} \;=\; \frac{1.100}{1.493} \;\approx\; 0.74
$$

仲間まで 0.39、隣まで 1.49——**隣は仲間の約4倍遠い**ので、0.74 という高い値になります。日本はこの分け方にしっくり収まっている、と数字が言っています。いっぽう新興と先進のあいだにいるガボンは $a_i \approx 1.49,\ b_i \approx 1.81$ とほぼ互角で $s_i \approx 0.18$。イラクにいたっては最も近い別クラスタのほうがわずかに近く $s_i \approx -0.07$ と**負**——「グループ0に入れたが、じつはグループ1寄り」という揺れを、この符号が拾っています（全142か国のうち $s_i$ が負なのは3か国だけです）。

**全体のスコアは各国の平均**　`silhouette_score` が返すのは、この $s_i$ を全国で平均した一つの数です（K=3 では約 0.513）。クラスタ別に平均すると、先進グループが 0.67 と飛び抜けて高く、最貧 0.45・新興 0.46 と続きます。先進国は「豊かで長寿」という一点に固まりやすく、いちばん**くっきりまとまっている**からです。慣性が「中はぎゅっと」だけを見たのに対し、シルエットは「外とはくっきり」も同時に見る——だから K を選ぶ物差しとして、より頼りになります。


**符号が負になるとき**　$s_i<0$ は $b_i<a_i$、つまり自分のクラスタの仲間より、最も近い別クラスタのほうが平均的に近い状態です。割り当てが最善でない疑いのサインで、境界がにじむ現実データでは少数出るのがふつう（ここでは3か国）。逆に負が大量に出るなら、K の選び方か手法そのものを疑うべき警告灯になります。各点の $s_i$ をクラスタごとに高い順で棒に並べた**シルエット図**を描くと、どのクラスタが厚く（収まりが良く）どのクラスタが薄いか、負の点がどこに潜むかが一目で分かり、全体スコアという一つの数だけでは見えない内訳を読めます。下のセルで、日本の $a_i,\ b_i,\ s_i$ を手計算し、`silhouette_samples` の値とぴたり一致することを確かめます。

In [ ]:
# 深掘りの数値確認：日本のシルエット係数を手計算し、silhouette_samples と一致することを見る
from sklearn.metrics import silhouette_samples
from sklearn.metrics.pairwise import euclidean_distances
_d = df[df.year == 2007].copy().reset_index(drop=True)
_d['loggdp'] = np.log10(_d['gdpPercap'])
_X = StandardScaler().fit_transform(_d[['lifeExp', 'loggdp']])
_lab = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(_X)
_D = euclidean_distances(_X)
i = _d.index[_d.country == 'Japan'][0]
ci = _lab[i]; same = (_lab == ci); same[i] = False
a_i = _D[i, same].mean()
b_i = min(_D[i, _lab == c].mean() for c in range(3) if c != ci)
print(f'日本: a_i={a_i:.3f}  b_i={b_i:.3f}  s_i=(b-a)/max(a,b)={(b_i-a_i)/max(a_i,b_i):.3f}')
print(f'silhouette_samples の値       : {silhouette_samples(_X, _lab)[i]:.3f}  <- 一致')

### 深掘り：慣性はなぜ必ず下がるのか、そしてエルボーとシルエットが指す K は違ってよい

**慣性は K を増やすほど必ず下がる**　TODO② の慣性を K=2 から 8 へ並べると、$84.2 \to 53.8 \to 37.1 \to 25.1 \to 19.2 \to 16.0 \to 13.8$ と、**一度も反転せず**下がり続けます。理由は単純で、クラスタを一つ増やせば、いちばん散らばっている塊を割って中心を増やせるので、距離二乗和は必ず減るからです。極端には、国の数だけクラスタを作れば各点が自分の中心になり、慣性はゼロ。だから「慣性が最小の K」を選ぶのは無意味で、**下がり幅が急に緩む折れ曲がり（肘）**を読むのがエルボー法です。

**肘はどこか**　慣性の「前の K からの下がり幅」を見ると、$K{=}3$ で $30.5$、$K{=}4$ で $16.6$、$K{=}5$ で $12.0$、$K{=}6$ で $5.9$。K=3 までは大きく削れ、それ以降は削り幅がだらだらと小さくなります。この「急な下りが緩やかに変わる」曲がり角が **K=3** のあたりで、散布図で見た発展の三段階と一致します。ただしエルボーは目視で決める柔らかい目安で、人によっては K=4 に見えることもある——数値の裏づけはあっても、**主観の残る指標**だと承知しておきます。

**シルエットのピークは K=2、そして本当に食い違う**　いっぽうシルエットスコアを K=2〜8 で並べると次のとおりです（実行値）。

| K | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|
| シルエット | **0.599** | 0.513 | 0.479 | 0.514 | 0.490 | 0.474 | 0.461 |

**ピークははっきり K=2**（0.599）。「豊かな国」と「そうでない国」の二分が、いちばんくっきり分かれるという結果です。エルボーの K=3 とシルエットの K=2 は、**指している K が違います**。しかも K=3 以降のシルエットは 0.513、0.479、0.514… とほぼ横ばいで、K=5（0.514）が K=3（0.513）をごくわずかに上回りさえします。つまりシルエットは「2 が最良、それ以外は似たり寄ったり」と言っており、**3 を積極的に推してはいない**のです。

**食い違ったらどう決めるか（つまずきどころ：K の恣意性）**　ここが本題です。指標が割れるのは現実データの常で、どちらかが間違いというわけではありません。二分割は“分離”こそ最良ですが、途上国と新興国をひとまとめにしてしまう。三分割は分離では二番手でも、**最貧・新興・先進という語りたい構造**を捉えます。**最後は「何を説明したいか」という目的が決める**——今回は発展の三段階を見たいので K=3 を採ります。「なぜその K か」と問われたら、「シルエットは2を推すが、途上国と新興国を分けて論じたいので、解釈可能性を優先して3にした」と**根拠つきで言える**こと。それがこの回のゴールです。K に唯一の正解はなく、指標は羅針盤であって裁判官ではない、と心得ておきましょう。


**ほかにも K を選ぶ物差しはある**　エルボーとシルエットのほかにも、ギャップ統計量（乱数データと比べて“不自然にまとまっている”K を探す）や、Calinski–Harabasz 指数・Davies–Bouldin 指数といった内部指標があります。どれも凝集と分離の別々の測り方で、**指標が違えば推す K も微妙に変わる**のがふつうです。だからこそ一つの指標を絶対視せず、複数を見比べて最後は目的で決める姿勢が要ります。

**Calinski–Harabasz と Davies–Bouldin は何を測っているのか**　名前だけでは中身が分からないので、測っているものの違いだけ押さえます。**Calinski–Harabasz 指数**は、クラスタ**間**のばらつき（中心どうしが全体の重心からどれだけ離れているか）を、クラスタ**内**のばらつき（各点が自分の中心からどれだけ離れているか＝慣性）で割った**比**です（クラスタ数と国数で目盛りを合わせたうえで割ります）。離れているほど分子が大きく、まとまっているほど分母が小さいので、**大きいほどよい**。全体を一つの比に均した物差しです。いっぽう**Davies–Bouldin 指数**は、クラスタを一つずつ取り上げ、相手ごとに「自分の広がり＋相手の広がり」を「中心どうしの距離」で割り、その中の**いちばん悪い（大きい）相手**の値を採ります。それを全クラスタで平均したものが指数で、重なりが小さいほど下がるので**小さいほどよい**。

**だから推す K がずれる**　違いは「平均を見るか、最悪を見るか」に尽きます。Calinski–Harabasz は全体を一つの比に均すので、大多数のクラスタがきれいなら、一組くらい紛らわしいペアがあっても値は高いまま保たれます。Davies–Bouldin は各クラスタの**いちばん困った相手**だけを見るので、その一組があるだけで値が悪化します。シルエットはさらに別で、点ごとに凝集と分離を測ってから平均します。同じ「凝集と分離」を、**比で見るか・最悪ペアで見るか・点ごとに見るか**が違う。エルボーとシルエットのあいだで起きたことは、内部指標どうしのあいだでも起きます——推す K が指標ごとに少しずつずれる理由は、この測り方の違いにあります。各指標の厳密な定義は、クラスタ分析だけを主題にした専門書（Everitt ら『Cluster Analysis』）の、妥当性指標を並べた章に譲ります。

**なぜ図を見るだけでは足りないのか**　今回は平均寿命と GDP の2項目だけなので散布図に描け、目で「3つっぽい」と見当がつきました。しかし指標がもっと多く、4次元・10次元…と増えれば、**そもそも図に描けません**。目視が効かない高次元でこそ、エルボーやシルエットのような**数値で K を語る物差し**が要ります。「図を見たら3つに見えた」で通用するのは低次元の幸運なケースだけ、と割り切っておきましょう。

In [ ]:
# TODO: K=2〜8 で、慣性とシルエットスコアを計算して表示してください
# ヒント: KMeans(n_clusters=k, random_state=42, n_init=10).fit(X) の .inertia_ が慣性
# ヒント: silhouette_score(X, km.labels_) がシルエット。高いほど良い
...

## 3. 外部指標 ARI（大陸と比べる）

クラスタリングの結果が、大陸（continent）とどれくらい一致するかを ARI で測ります。

### 深掘り：ラベルなしの評価はなぜ難しいか／ARI・DBSCAN・階層という別の道具

**内部指標と外部指標**　分類（第8回）なら「正解ラベルと合っているか」で正解率を測れました。ところがクラスタリングは正解ラベルを使わないのが身上——**採点する答えがそもそもない**のです。だから評価は二段構えになります。慣性やシルエットのように**データの形の良さだけ**で測るのが**内部指標**。いっぽう、もし別途「外の分類（classification）」があれば、それとの一致で測れるのが**外部指標**で、ここで使う ARI（調整ランド指数）がその代表です。内部指標は「まとまりの良さ」は言えても「意味があるか」は言えず、外部指標は「既知の分類と重なるか」は言えても、比べる分類がなければ使えません。ラベルなしの評価が難しいのは、この**唯一の物差しが存在しない**ことに尽きます。

**ARI を読む**　このデータには「大陸（continent）」という外の分類がついています。K=3 の結果と大陸の一致を測ると **ARI ≈ 0.36**。ARI は 1 で完全一致・0 で偶然の分け方と同程度なので、0.36 は**弱い一致**です。アフリカの多くは最貧グループに入って大陸と重なる一方、アジアやアメリカ大陸は途上国も先進国も入り混じります——日本とアフガニスタンは同じアジアでも別グループです。**発展度は地理では決まらない**ことを、ARI が一つの数で語っています。なお ARI が「調整」ランド指数と呼ばれるのは、偶然の一致分を差し引いてあるからで、でたらめな分け方なら 0 付近になるよう補正されています（単なる一致率だと、たまたま当たる分で下駄をはいてしまう）。

**発展①：形が変なら DBSCAN**　K-means が丸いかたまりしか囲えないのは、上で見たとおり中心からの距離で割り当てるからでした。**DBSCAN** は発想が違い、「点が**密に**つながっている領域」を一つのクラスタとみなします。密につながってさえいれば三日月でも渦巻きでも追いかけられ、しかも**クラスタ数 K を指定しなくてよい**（代わりに密度のしきい値を与えます）。どこの密集にも属さない点を「ノイズ（外れ値）」として自動で弾けるのも利点です。形が非球状のとき、K の決め方に悩むとき、外れ値（outlier）を混ぜたくないときの有力な代替になります。

**発展②：木で眺める階層クラスタリング（hierarchical clustering）**　もう一つの別路線が**階層クラスタリング**です。近い国から順に合体させ、その合体の歴史を**デンドログラム（dendrogram、木）**に描きます。K-means が「先に K を決めて一発で平らに分ける」のに対し、こちらは木を**好きな高さで横に切る**だけで、後から粒度（K）を選べます。高く切れば少数の大グループ、低く切れば多数の小グループ。K をいくつにするか迷うデータで、まず木を眺めて構造の見当をつける、という使い方に向きます（合体ルールは、まとまりが崩れにくいウォード法が無難です）。

**まとめ：手法も K も「データの形と目的」で選ぶ**　クラスタリングに万能の一手はありません。丸い塊なら K-means が速くて素直、非球状なら DBSCAN、構造をじっくり眺めたいなら階層。評価も、内部指標で形の良さを、外部指標（あれば）で既知分類との重なりを測り、そして最後は**目的**で締めます。「なぜこの手法・この K なのか」を根拠つきで語れること——それが、ラベルなき世界で迷子にならないための羅針盤です。


**DBSCAN の中身をもう少し**　DBSCAN は二つのつまみ——半径 $\varepsilon$（イプシロン）と、その半径内に必要な最小点数——で密度を定義します。半径内に十分な仲間を持つ点を「コア点」、コアの近くにいる点を「境界点」、どちらでもない孤立点を「ノイズ」と三分し、コア点どうしが近ければ同じクラスタへ数珠つなぎにします。この“つながり”で広げるため丸くない形も追える一方、密度がまばらなデータや、$\varepsilon$ の選び方には敏感です。

**階層クラスタリングの合体ルール**　階層クラスタリングは「どの二グループを近いとみなすか」の**リンケージ**を選べます。ウォード法（Ward's method）は合体後のまとまりの崩れが小さい組を選び（K-means と相性がよく、丸い塊向き）、最長距離法はグループ間のいちばん遠いペアで測り（コンパクトな塊を作りやすい）、平均距離法は全ペアの平均で測ります。ルール次第で木の形が変わるので、まずウォード法、が無難な出発点です。

**距離をどう測るかも選べる**　DBSCAN も階層クラスタリングも、根っこは「点と点の距離」です。今回は素直なユークリッド距離ですが、指標の性質によってはマンハッタン距離（Manhattan distance）や、相関を距離に読み替える方法など、**距離の定義そのものを選ぶ**余地もあります。どんな距離で「似ている」を測るかは、標準化と並んで結果を左右する設計判断——手法を選ぶ前に、まず「この問題での“近い”とは何か」を決めるのが順序です。

In [ ]:
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(d['continent'], d['cluster'])
print(f'クラスタと大陸の一致度 ARI: {ari:.3f}')
print('→ 1で完全一致、0でランダム。発展のグループは地理と完全には重ならない')

## 目標に答えられたか

- 今日の目標は「国々を、根拠あるグループ数で、意味あるグループに分けたい」でした
- TODO①で、3つのグループは「最貧・新興・先進」に対応していましたか？ 代表的な国は？
- TODO②で、エルボーの肘とシルエットのピークは、同じKでしたか？ 食い違ったとき、どう決めますか？
- ARI（大陸との一致）は高かったですか、低かったですか？ それは何を意味するでしょう？
- 「なぜそのKなのか」を、あなたなら報告でどう説明しますか？

## 今日の要点

- クラスタリングに正解ラベルはない。だから採点ではなく、「凝集（中はぎゅっと）」と「分離（外とはくっきり）」という形の良さを測るしかない
- K-means の4ステップは、慣性を下げ続ける手続き。割り当ても中心更新もどちらも慣性を減らすだけだから、必ずどこかで止まる
- 距離で似ているかを決める手法なので、標準化と対数化を先に済ませないと、桁の大きい GDP だけで結論が決まってしまう
- 慣性は K を増やせば必ず下がる。だから最小値を探すのではなく、下がり幅が緩む折れ曲がり（肘）を読む
- シルエットは凝集と分離を同時に見るが、この142か国では K=2 を推した。エルボーの K=3 とは食い違う
- 指標が割れるのは異常ではなく、指標ごとに測っているものが違うから。最後は「何を説明したいか」が K を決める
- クラスタ番号はただの名札。中身を要約して「最貧・新興・先進」と名づけるまでが分析で、そこは人間の仕事


## 次回へ

第11回と第12回は列（変数）の方向に、今日は行（個体）の方向にデータを要約しました。教師なし学習の二本柱を、これで一周したことになります。ただし今日成り立っていた前提を見落とさないでください。平均寿命と GDP という**数値**だったからこそ、標準化して距離を測り、平均で中心を求められた——K-means も、慣性も、シルエットも、足し算と割り算ができる数字の上に建っています。

次回は、その前提が外れます。相手はカテゴリ、つまり質的データです。ジャンルや年代のような選択肢の集まりには、平均も距離もそのままでは定義できません。それでも「どのカテゴリとどのカテゴリが近いか」を一枚の平面に置いて見せるのがコレスポンデンス分析（対応分析）です。数値データの地図を作ってきた最後に、質的データの地図を作ります。


## 課題（提出）

**提出するもの**: 応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

1節では K-means で142か国を3グループに分けました。今度は、3節の深掘りで触れた**階層クラスタリング**（ウォード法）で同じことをします。

`from sklearn.cluster import AgglomerativeClustering` を読み込み、`AgglomerativeClustering(n_clusters=3)` で同じ `X` を3グループに分けてください。結果は `d['hc']` のように**別の列**に入れます（`d['cluster']` は応用②で使うので上書きしないでください）。そのうえで、TODO①と同じ要約表（国数・平均寿命の平均・GDPの平均）を表示してください。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

1節の `KMeans(n_clusters=3, random_state=42, n_init=10)` を `AgglomerativeClustering(n_clusters=3)` に置き換えるだけです。`.fit_predict(X)` でラベルが返ります（ウォード法が既定なので、引数はそれだけで構いません）。
要約表は TODO① のコードの `groupby('cluster')` を `groupby('hc')` に変えます。

</details>


### 応用②（判断）

応用①の階層クラスタリングの結果と、1節の K-means の結果（`d['cluster']`）は、どれくらい一致していますか。2つの結果の ARI を **小数第2位まで** で答えてください（例: 0.57）。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

3節と同じ `adjusted_rand_score` に、`d['continent']` の代わりに `d['cluster']` を渡します。2つのラベルはどちらを先に渡しても同じ値です。
`pd.crosstab(d['cluster'], d['hc'])` を表示すると、どのグループの何か国が違ったかも分かります。

</details>


### 応用③（解釈）

K-means と階層クラスタリングは、142か国の大部分で同じ分け方をしましたが、一部の国で割り当てが違いました。応用②の ARI と、2つの要約表の違い（どのグループが増え、どのグループが減ったか）をふまえ、報告書に**どちらの結果を載せるか**、その理由を、**国際協力の担当者**に向けて3行程度で書いてください。


（ここに3行程度で書く）


<details><summary>詰まったら</summary>

どちらが「正しい」かは決められません（採点する正解ラベルがないからです）。3節の深掘りで見た「目的で決める」という考え方と、手法によって所属が変わった国をどう扱うかを書きます。

</details>


## 発展（任意）

### HDBSCAN——K を決めず、まとまらない国は外れ値にする

K-means には、人が決めなければならないことが二つありました。K を先に決めることと、**すべての国をどこかのグループに必ず入れる**ことです。エルボーとシルエットで K の根拠は探せますが、境界の国（シルエットが負だったイラクなど）まで無理にどこかへ押し込むのは変わりません。

**HDBSCAN** は発想が違います。3節の深掘りで触れた DBSCAN と同じく「点が密に集まっている場所」をクラスタとみなし、さらに密度のしきい値（半径 ε）を一つに固定せず、密度を変えながら安定して残るまとまりを階層的に選びます。だから K も ε も指定しません。決めるのは主に「最低何か国でひとまとまりと認めるか」（`min_cluster_size`）です。

どの密集にも属さない国はラベル -1（外れ値）として残します。全点をどこかに入れる K-means より、境界のにじむ実データではこの方が正直なことが多いです。scikit-learn 1.3 以降に `sklearn.cluster.HDBSCAN` として入っているので、追加のインストールは要りません。

**「密度のしきい値を動かす」とは何をしているのか**　DBSCAN は半径 $\varepsilon$ を一つ決め、その値で「密」か否かを切りました。$\varepsilon$ を大きいほうから少しずつ縮めていくと、はじめは全部つながっていた大きな塊がやがて割れ、さらに縮めると芯だけが残り、最後はばらばらの点になります。「どのしきい値で、どの塊が、どの塊に割れたか」を記録すると、塊が塊を含む**入れ子の木**ができます。HDBSCAN がやるのは、この木の枝のうち「$\varepsilon$ を広い範囲で動かしても形を変えずに残る枝」＝**長く生き延びた枝**を選び出すことです。しきい値を一つに決め打ちする代わりに、**しきい値の変化に対して安定していること**を選択の基準に置き換えた——だから $\varepsilon$ も K も指定せずに済み、どの枝にも最後まで残らなかった点が外れ値として手元に残ります。密度の階層をどう作り、その木からどの枝を選ぶのかという手続きの厳密な話は、アルゴリズムの中身を段階を追って解説した実装ライブラリのドキュメント（"How HDBSCAN Works"）に譲ります。


In [ ]:
from sklearn.cluster import HDBSCAN   # scikit-learn 1.3 以降に同梱
hdb = HDBSCAN(min_cluster_size=8)
d['hdb'] = hdb.fit_predict(X)
n_cl = int(d['hdb'].max()) + 1
n_out = int((d['hdb'] == -1).sum())
print(f'クラスタ数: {n_cl}、外れ値（ラベル -1）: {n_out}か国 / {len(d)}か国')
print(d.groupby('hdb').agg(国数=('country', 'size'), 平均寿命=('lifeExp', 'mean'), GDP=('gdpPercap', 'mean')).round(0).to_string())
print('外れ値の例:', d[d['hdb'] == -1]['country'].head(12).tolist())


In [ ]:
colors_h = ['#e63946', '#e9c46a', '#2a9d8f', '#457b9d', '#8d99ae']
for cl in range(n_cl):
    m = d['hdb'] == cl
    plt.scatter(d.loc[m, 'gdpPercap'], d.loc[m, 'lifeExp'], c=colors_h[cl % len(colors_h)], s=30, alpha=0.8, label=f'クラスタ{cl}')
m = d['hdb'] == -1
plt.scatter(d.loc[m, 'gdpPercap'], d.loc[m, 'lifeExp'], marker='x', c='gray', s=40, label='外れ値')
plt.xscale('log'); plt.xlabel('一人あたりGDP（対数）'); plt.ylabel('平均寿命'); plt.legend()
plt.tight_layout(); plt.show()


**読み方**　結果は3つのクラスタ（30・39・32か国）と、41か国の外れ値になりました。クラスタの中身は K-means の最貧・新興・先進とほぼ重なりますが、それぞれの「芯」だけが残り、境界の国は外れ値に回されています。

グラフで×印を見ると、密集から外れた場所に散っています。GDP は高いのに平均寿命が短いガボン・ボツワナ・赤道ギニア・南アフリカ（資源国や HIV の影響）、逆に GDP のわりに長寿のチリ・コスタリカ・キューバ、平均寿命が特に短いスワジランド・ジンバブエなどで、「発展の三段階」の物語に収まらない国です。K-means はこれらも無理にどこかへ入れます。シルエットが負だったイラクやトリニダード・トバゴも、HDBSCAN では外れ値に回っています。HDBSCAN は「まとまらないものはまとめない」と言います。

外れ値が142か国中41か国（約3割）というのは多く感じるかもしれません。`min_cluster_size` は「密」の基準とクラスタの選び方の両方に効くため、値を変えても外れ値数は単純に増減しません（5 にすると5クラスタ・外れ値43か国、10 にすると2クラスタ・外れ値31か国）。値しだいでクラスタ数も動くので、K-means の K と同じく「なぜその値か」は自分で説明できるようにしておきます。

試すなら、`min_cluster_size` を 5 や 10 に変えて、クラスタ数と外れ値の顔ぶれがどう動くか見てください。
